# Stage 2 — Distill the 3B student with FSDP + LoRA

This notebook fine-tunes Qwen2.5-VL-3B on the Stage 1 teacher parquet using Ray Train + FSDP + LoRA. The vision tower stays frozen (standard [LLaVA recipe](https://arxiv.org/abs/2304.08485)); only the language model gets LoRA adapters, so trainable parameters drop to ~1% of the model. FSDP `FULL_SHARD` distributes optimizer state across the four GPUs at bf16 mixed precision.

The output is a ~100 MB adapter directory that drops into Stage 3 as a model swap. The notebook scales every knob (rows, workers, epochs) down so the loop finishes in minutes on a workspace cluster; [`scripts/run_distill_student_lora.py`](../scripts/run_distill_student_lora.py) runs the same pipeline at production scale. For the end-to-end story across all three stages, see the template's [`README.ipynb`](../README.ipynb).

In [ ]:
import os, sys, json
import ray

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

# If the cluster is already up with stale runtime_env (e.g. missing peft on
# workers), tear it down so the new pip list below actually takes effect.
if ray.is_initialized():
    ray.shutdown()

ray.init(
    runtime_env={
        "working_dir": os.path.abspath(os.path.join(os.getcwd(), "..")),
        # peft + accelerate aren't in the workspace base image; install on
        # workers so train_loop_per_worker can `from peft import ...`.
        "pip": ["peft>=0.12", "accelerate>=0.34"],
    },
)
print("Cluster resources:", json.dumps(ray.cluster_resources(), indent=2))

## Configure the run

The data pipeline and training loop import from [`scripts/run_distill_student_lora.py`](../scripts/run_distill_student_lora.py), so the notebook exercises the same code paths as a production job. Only row count, worker count, and epoch count are scaled down here.

In [ ]:
# Production helpers — same code paths as the submitted job
from scripts.run_distill_student_lora import (
    BuildSFTExample, train_loop_per_worker,
    fetch_and_resize_images, _split_bucket, _strip_code_fence,
    STUDENT_MODEL_ID, MAX_SEQ_LEN, MIN_PIXELS, MAX_PIXELS,
    LORA_R, LORA_ALPHA, LORA_DROPOUT, LORA_TARGET_MODULES,
)
from ray.train import ScalingConfig, RunConfig, CheckpointConfig
from ray.train.torch import TorchTrainer

# Smoke-scale knobs — small enough to run end-to-end in ~10 min on 2× L4.
N_SMOKE = 100
NUM_WORKERS_SMOKE = 2
NUM_EPOCHS_SMOKE = 1
PER_DEVICE_BS_SMOKE = 1
GRAD_ACCUM_SMOKE = 4         # effective batch = 1 × 2 × 4 = 8
LR_SMOKE = 1e-4
WARMUP_RATIO_SMOKE = 0.0     # tiny run, skip warmup

BASE_DIR = "/mnt/cluster_storage/vlm-distillation-catalog-enrichment"
TEACHER_PARQUET    = f"{BASE_DIR}/teacher_7b_enriched_10000.parquet"
SMOKE_CACHE_PATH   = f"{BASE_DIR}/sft_cache_smoke_{N_SMOKE}.parquet"
SMOKE_ADAPTER_DIR  = f"{BASE_DIR}/qwen25vl_3b_enrichment_lora_smoke"
SMOKE_RUN_DIR      = f"{BASE_DIR}/qwen25vl_3b_enrichment_runs_smoke"

if not os.path.exists(TEACHER_PARQUET):
    raise FileNotFoundError(
        f"Teacher parquet not found at {TEACHER_PARQUET}. "
        f"Run scripts/run_teacher_batch_label.py first."
    )

print(f"smoke config: N={N_SMOKE} workers={NUM_WORKERS_SMOKE} "
      f"epochs={NUM_EPOCHS_SMOKE} effective_batch={PER_DEVICE_BS_SMOKE * NUM_WORKERS_SMOKE * GRAD_ACCUM_SMOKE}")

## Build the SFT cache

Read the teacher parquet, drop rows whose teacher output isn't valid JSON, assign a deterministic train/val/test split, fetch each image once, and write a self-contained training parquet. The image bytes live in this cache so epochs 2..N never go back to HTTP.

In [ ]:
def _teacher_output_is_valid(row):
    try:
        obj = json.loads(_strip_code_fence(row["raw_output"]))
        return all(k in obj for k in ("category", "attributes", "search_tags", "description"))
    except Exception:
        return False

def _attach_split(row):
    row["split"] = _split_bucket(row["id"])
    return row

if os.path.exists(SMOKE_CACHE_PATH):
    cached = ray.data.read_parquet(SMOKE_CACHE_PATH)
    print(f"[cache] reusing {SMOKE_CACHE_PATH}")
else:
    ds = ray.data.read_parquet(TEACHER_PARQUET)
    ds = ds.filter(_teacher_output_is_valid).limit(N_SMOKE).map(_attach_split)
    ds = ds.map_batches(
        fetch_and_resize_images,
        batch_size=16,
        concurrency=8,
        batch_format="numpy",
    )
    ds.write_parquet(SMOKE_CACHE_PATH)
    cached = ray.data.read_parquet(SMOKE_CACHE_PATH)

n_train = cached.filter(lambda r: r["split"] == "train").count()
n_val = cached.filter(lambda r: r["split"] == "val").count()
n_test = cached.filter(lambda r: r["split"] == "test").count()
print(f"[split] train={n_train}  val={n_val}  test={n_test}")

## Sanity check one row

Confirm the teacher target parses, the image decodes, and the split assignment looks right. Any data issue surfaces here before the GPUs spin up.

In [ ]:
from PIL import Image
import io as _io

row = cached.take(1)[0]
print("id:        ", row["id"])
print("split:     ", row["split"])
print("title:     ", row["title"][:90])
print("teacher:   ", _strip_code_fence(row["raw_output"])[:240])
img = Image.open(_io.BytesIO(row["image_bytes"]))
img.thumbnail((256, 256))
img

## Build SFT examples

`BuildSFTExample` applies the Qwen2.5-VL chat template and processor to each row, producing fixed-shape numpy columns (`input_ids`, `labels`, `attention_mask`, `pixel_values`, `image_grid_thw`) that Ray Data streams into the trainer. Loss masking on the user prefix means only the assistant's JSON tokens contribute to the loss — image and prompt tokens are `-100`.

In [ ]:
build_kwargs = dict(
    fn_constructor_kwargs={
        "model_id": STUDENT_MODEL_ID,
        "max_length": MAX_SEQ_LEN,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
    },
    num_cpus=2,
    concurrency=4,
)

# Repartition before BuildSFTExample so the OutputSplitter at the trainer
# end has multiple blocks to balance across ranks. The smoke cache parquet
# is a single block; without this, OutputSplitter[split(2, equal=True)] can
# stall one rank waiting for the splitter to materialize, which on FSDP
# manifests as a NCCL collective hang at the first all-gather.
train_ds = (cached.filter(lambda r: r["split"] == "train")
                  .repartition(NUM_WORKERS_SMOKE * 2)
                  .map(BuildSFTExample, **build_kwargs))
val_ds = (cached.filter(lambda r: r["split"] == "val")
                .repartition(NUM_WORKERS_SMOKE * 2)
                .map(BuildSFTExample, **build_kwargs))

# Inspect one tokenized example. The shapes here drive every batching
# decision in the trainer; if anything's variable across rows, training will
# fail on the first stack() call.
example = train_ds.take(1)[0]
for k, v in example.items():
    if hasattr(v, "shape"):
        print(f"  {k:18} shape={tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:18} = {v!r}")

# Loss-mask sanity check: assistant tokens are non-(-100), prefix tokens are
# all -100. The fraction non-masked ≈ length(target_json) / MAX_SEQ_LEN.
import numpy as np
labels = example["labels"]
n_loss_tokens = int((labels != -100).sum())
print(f"\nloss-mask: {n_loss_tokens}/{len(labels)} tokens contribute to loss "
      f"({100 * n_loss_tokens / len(labels):.1f}%)")

## Run training

`TorchTrainer` runs `train_loop_per_worker` across the GPU workers. Each worker loads the 3B base model, applies LoRA, FSDP-wraps the language model, and streams the SFT shards through. On 2× L4 at smoke scale this typically finishes in 8–12 minutes — most of which is the first-step compile and model load.

If the run OOMs, drop `MAX_SEQ_LEN` to 1024 or set `PER_DEVICE_BS_SMOKE = 1` and `GRAD_ACCUM_SMOKE = 8`.

In [ ]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "model_id": STUDENT_MODEL_ID,
        "lr": LR_SMOKE,
        "num_epochs": NUM_EPOCHS_SMOKE,
        "per_device_bs": PER_DEVICE_BS_SMOKE,
        "grad_accum": GRAD_ACCUM_SMOKE,
        "warmup_ratio": WARMUP_RATIO_SMOKE,
        "weight_decay": 0.01,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "train_size": n_train,
        "adapter_dir": SMOKE_ADAPTER_DIR,
        # Production uses 50; smoke has only ~10 grad steps total, so drop
        # to 5 to demonstrate that step-level checkpointing actually fires.
        # Expect ckpts at step 5 and step 10 (the latter coincides with the
        # end-of-epoch ckpt, so retention picks whichever has lower val_loss).
        "save_every_n_steps": 5,
        # Eval bs > train bs: no backward, no grad checkpoint tax. Smoke val
        # is ~8 rows, sharded across 2 workers → 4 rows each → exactly 1
        # batch per rank at eval_bs=4. Production uses 4 for the same reason.
        "eval_per_device_bs": 4,
        # Reproducibility: torch + cuda seeds offset per-rank, random/numpy
        # uniform. Without this, dropout and any python-side randomness drift
        # run-to-run even with identical data and config.
        "seed": 42,
    },
    scaling_config=ScalingConfig(
        num_workers=NUM_WORKERS_SMOKE,
        use_gpu=True,
        accelerator_type="L4",
        resources_per_worker={"GPU": 1, "CPU": 4},
        # PACK both workers onto the same node so FSDP all-gathers /
        # reduce-scatters use intra-node shared memory instead of crossing
        # the cluster network. Without this, a workspace with only
        # single-GPU L4 nodes available will land the 2 workers on
        # separate nodes; the first NCCL collective then either crawls
        # (10x+ slower per step) or hangs at all-gather. Requires a node
        # with >= num_workers GPUs (e.g. g6.12xlarge with 4x L4).
        # placement_strategy="PACK",
    ),
    run_config=RunConfig(
        storage_path=SMOKE_RUN_DIR,
        # num_to_keep=3 so all of (step5, step10, epoch0) can co-exist for
        # the demo — easy to inspect each in result.best_checkpoints.
        checkpoint_config=CheckpointConfig(
            num_to_keep=3,
            checkpoint_score_attribute="val_loss",
            checkpoint_score_order="min",
        ),
    ),
    datasets={"train": train_ds, "val": val_ds},
)
result = trainer.fit()
print(f"\n[done] metrics: {result.metrics}")
print(f"[done] best checkpoint: {result.checkpoint}")
print(f"[done] adapter: {SMOKE_ADAPTER_DIR}")

## Verify the adapter

Confirm the saved directory has the expected files (`adapter_config.json` plus the LoRA weights) and that the config records the LoRA rank and target modules used during training. The full inference test belongs in Stage 3, which loads this adapter into vLLM at serve time.

In [ ]:
import json as _json

files = sorted(os.listdir(SMOKE_ADAPTER_DIR))
print("adapter directory contents:")
for f in files:
    p = os.path.join(SMOKE_ADAPTER_DIR, f)
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {f:40} {size_mb:>8.2f} MB")

with open(os.path.join(SMOKE_ADAPTER_DIR, "adapter_config.json")) as fh:
    adapter_cfg = _json.load(fh)
print("\nadapter_config.json (key fields):")
for k in ("r", "lora_alpha", "lora_dropout", "target_modules",
         "task_type", "base_model_name_or_path"):
    print(f"  {k:28} = {adapter_cfg.get(k)!r}")

## Submit the full run as an Anyscale Job

Once the smoke run looks healthy, submit the full 10k-row training as a job — same compute config as the workspace, no code changes:

```bash
anyscale job submit --config-file ../job_config.yaml \
  --entrypoint "python scripts/run_distill_student_lora.py" \
  --env HF_TOKEN=$HF_TOKEN
```

Then load the trained adapter into the Stage 3 inference pipeline by setting `QWEN_LORA_ADAPTER_DIR` to the adapter directory before running [`scripts/run_enrich_and_embed.py`](../scripts/run_enrich_and_embed.py).